In [1]:
import pandas as pd

Questions to answer

1. What proportion of projects are categorized as remote vs field?
2. What projects require a manual decision?
3. What projects are on the cusp and can go either way?
4. How does this compare to what we expected (30% field / 70% remote)

Other interesting analyses to show:
- bi-modal distribution of image availability. 67% of polygons have 0-4 images and 13% have 16-30 images.
- What is the decision for the largest projects?

In [10]:
gp = pd.read_parquet("../data/tm_raw/04-13-2026_tm.geoparquet")
missing = gp[gp.short_name == 'DRC_23_PFDK']

In [9]:
missing

,project_id,project_name,country,framework_key,short_name,cohort,site_name,site_id,poly_name,poly_uuid,...,distr,calc_area,status,version_name,is_active,source,deleted_at,geom,project_phase,ttc


In [4]:
gp.columns

Index(['project_id', 'project_name', 'country', 'framework_key', 'short_name',
       'cohort', 'site_name', 'site_id', 'poly_name', 'poly_uuid',
       'plantstart', 'practice', 'target_sys', 'distr', 'calc_area', 'status',
       'version_name', 'is_active', 'source', 'deleted_at', 'geom',
       'project_phase', 'ttc'],
      dtype='object')

In [18]:
# combine ttc
# 16,505 flagged as missing ttc
ttc = pd.read_csv("../data/ttc/20260413_ttc_c2.csv")
feats = pd.read_csv("../data/feats/tm_feats_c2_04-13-2026.csv")
dropped = pd.read_csv("../data/ttc/ttc_dropped_polygons_report.csv")
ttc.shape, feats.shape, dropped.shape

((25691, 13), (26034, 16), (8, 9))

In [19]:
list(set(feats.project_name))

['BUR_23_JR',
 'DRC_23_CCAO',
 'KEN_23_GROWTECH',
 'RWA_23_FHA',
 'KEN_23_PARAN',
 'GHA_23_SAFI',
 'RWA_23_PBC',
 'DRC_23_IADL',
 'KEN_23_LF',
 'RWA_23_RECOR',
 'GHA_23_KUKUOM',
 'GHA_23_TROPENBOS',
 'KEN_23_EDEN',
 'KEN_23_WWANC',
 'GHA_23_YM',
 'RWA_23_RDI',
 'BUR_23_PWP',
 'KEN_23_GROOTS',
 'RWA_23_FESD',
 'KEN_23_SESANA',
 'DRC_23_UESEF',
 'GHA_23_FANTEAKWA',
 'KEN_23_AAE',
 'RWA_23_OSEPCCA',
 'KEN_23_ITF',
 'KEN_23_SJT',
 'BUR_23_APRN-BEPB',
 'RWA_23_REDO',
 'RWA_23_BIRDLIFE',
 'KEN_23_WWF',
 'KEN_23_ENDAO',
 'GHA_23_ROCHA',
 'GHA_23_MIF',
 'KEN_23_KENVO',
 'BUR_23_UNIPROBA',
 'KEN_24_VOKENEL',
 'GHA_23_INEC',
 'GHA_23_GGV',
 'KEN_22_AFGL',
 'RWA_23_RWARRI',
 'RWA_23_RDIS',
 'KEN_23_ORCHARD',
 'RWA_23_AEE',
 'KEN_23_WEZESHA',
 'KEN_22_GREENPOT',
 'BUR_23_PVC',
 'RWA_23_BIOCOOR',
 'BUR_23_COH',
 'KEN_23_MOTO',
 'KEN_23_PACJA',
 'RWA_23_RCCDN',
 'GHA_23_SAKAM',
 'RWA_23_ARDE',
 'BUR_23_WDI',
 'DRC_23_PWP',
 'KEN_23_SKWT',
 'RWA_23_ROAM',
 'KEN_23_LOCCOG',
 'KEN_23_LWFL',
 'KEN_23_NA

In [27]:
26034-25691-8

335

In [17]:
feats.project_id.nunique()

93

In [30]:
feats.poly_id.nunique(), ttc.poly_id.nunique()

(26034, 25691)

In [19]:
#list(set(feats.project_name))

In [32]:
feats.head()

,cohort,project_id,poly_id,site_id,project_name,geometry,plantstart,practice,target_sys,area,ttc_2020,ttc_2021,ttc_2022,ttc_2023,ttc_2024,notes
0,terrafund-cohort-2,9c22bab7-9170-4502-bfd5-f7ea361764ce,5d0552bb-32e9-48f8-80bb-e6b404d8fcfd,9a2d475c-cb1d-449b-8304-7253909f65d9,BUR_23_ADIC,"POLYGON ((29.44554845297829 -3.14129515828657,...",2024-11-18,tree-planting,agroforest,2.802268,NaN,NaN,NaN,NaN,NaN,NaN
1,terrafund-cohort-2,9c22bab7-9170-4502-bfd5-f7ea361764ce,ecccaaa1-6b49-47e9-a304-f499319eaa65,9a2d475c-cb1d-449b-8304-7253909f65d9,BUR_23_ADIC,POLYGON ((29.460819107417706 -3.14556400414023...,2024-11-12,tree-planting,agroforest,3.067286,NaN,NaN,NaN,NaN,NaN,NaN
2,terrafund-cohort-2,9c22bab7-9170-4502-bfd5-f7ea361764ce,7ebcc9a3-36f7-4bca-8d7b-c1f5b45fe6e3,9a2d475c-cb1d-449b-8304-7253909f65d9,BUR_23_ADIC,POLYGON ((29.413067338538635 -3.15036862124662...,2024-11-18,tree-planting,agroforest,2.483303,NaN,NaN,NaN,NaN,NaN,NaN
3,terrafund-cohort-2,9c22bab7-9170-4502-bfd5-f7ea361764ce,d86fe98b-d8f9-44c9-8ee1-657cb0ce0996,9a2d475c-cb1d-449b-8304-7253909f65d9,BUR_23_ADIC,POLYGON ((29.415172457460713 -3.13156683378281...,2024-11-12,tree-planting,agroforest,2.269991,NaN,NaN,NaN,NaN,NaN,NaN
4,terrafund-cohort-2,9c22bab7-9170-4502-bfd5-f7ea361764ce,2cd5e8fc-ff2d-44fa-8e46-335a4402c79d,9a2d475c-cb1d-449b-8304-7253909f65d9,BUR_23_ADIC,"POLYGON ((29.4466503836931 -3.143557678754931,...",2024-11-20,tree-planting,agroforest,1.841360,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
# confirm years match 
feats.columns

Index(['cohort', 'project_id', 'poly_id', 'site_id', 'project_name',
       'geometry', 'plantstart', 'practice', 'target_sys', 'area', 'ttc_2020',
       'ttc_2021', 'ttc_2022', 'ttc_2023', 'ttc_2024', 'notes'],
      dtype='object')

In [15]:
ttc.pred_year.value_counts()

pred_year
2023    17343
2024     7888
2022      451
2021        8
2020        1
Name: count, dtype: int64

In [13]:
maxs = ttc[ttc.TTC == ttc.TTC.max()]
maxs.shape

(25, 13)

In [4]:
ttc.head()

,poly_id,TTC,area_HA,shift_error,small_site_error,lulc_lower_error,lulc_upper_error,subregion_lower_error,subregion_upper_error,error_plus,error_minus,plus_minus_average,pred_year
0,6dba36bc-1654-4a04-bc6b-449b09b8b8ad,0.000000,0.076398,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2020
1,22560785-182b-4c06-b5a5-f355eaf4f907,45.441629,1.888055,0.023921,0.0,0.000077,0.000110,0.000770,0.002145,0.552189,0.544634,0.548412,2021
2,a09e8e9f-f0f6-438a-95ac-aa5457082d46,0.000000,0.210312,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2021
3,fb738f83-65c2-44cd-9384-ecaa692aff95,26.835350,0.962189,0.243444,0.0,0.000131,0.000186,0.001304,0.003632,3.267903,3.266635,3.267269,2021
4,43489fa1-a448-4de7-bd69-e23e7b6d847c,42.696573,6.713105,0.079384,0.0,0.000082,0.000117,0.000820,0.002283,1.697526,1.695084,1.696305,2021


In [5]:
feats.head()

,cohort,project_id,poly_id,site_id,project_name,geometry,plantstart,practice,target_sys,area,ttc_2020,ttc_2021,ttc_2022,ttc_2023,ttc_2024,notes
0,terrafund-cohort-2,9c22bab7-9170-4502-bfd5-f7ea361764ce,5d0552bb-32e9-48f8-80bb-e6b404d8fcfd,9a2d475c-cb1d-449b-8304-7253909f65d9,BUR_23_ADIC,"POLYGON ((29.44554845297829 -3.14129515828657,...",2024-11-18,tree-planting,agroforest,2.802268,NaN,NaN,NaN,NaN,NaN,missing-ttc
1,terrafund-cohort-2,9c22bab7-9170-4502-bfd5-f7ea361764ce,ecccaaa1-6b49-47e9-a304-f499319eaa65,9a2d475c-cb1d-449b-8304-7253909f65d9,BUR_23_ADIC,POLYGON ((29.460819107417706 -3.14556400414023...,2024-11-12,tree-planting,agroforest,3.067286,NaN,NaN,NaN,NaN,NaN,missing-ttc
2,terrafund-cohort-2,9c22bab7-9170-4502-bfd5-f7ea361764ce,7ebcc9a3-36f7-4bca-8d7b-c1f5b45fe6e3,9a2d475c-cb1d-449b-8304-7253909f65d9,BUR_23_ADIC,POLYGON ((29.413067338538635 -3.15036862124662...,2024-11-18,tree-planting,agroforest,2.483303,NaN,NaN,NaN,NaN,NaN,missing-ttc
3,terrafund-cohort-2,9c22bab7-9170-4502-bfd5-f7ea361764ce,d86fe98b-d8f9-44c9-8ee1-657cb0ce0996,9a2d475c-cb1d-449b-8304-7253909f65d9,BUR_23_ADIC,POLYGON ((29.415172457460713 -3.13156683378281...,2024-11-12,tree-planting,agroforest,2.269991,NaN,NaN,NaN,NaN,NaN,missing-ttc
4,terrafund-cohort-2,9c22bab7-9170-4502-bfd5-f7ea361764ce,2cd5e8fc-ff2d-44fa-8e46-335a4402c79d,9a2d475c-cb1d-449b-8304-7253909f65d9,BUR_23_ADIC,"POLYGON ((29.4466503836931 -3.143557678754931,...",2024-11-20,tree-planting,agroforest,1.841360,NaN,NaN,NaN,NaN,NaN,missing-ttc


In [33]:
## things to figure out
# move notes to end and retest -

# Keep only the columns needed from ttc (drop error columns)
ttc = ttc[['poly_id', 'TTC', 'pred_year']]

# Pivot ttc to wide format: one row per poly_id, one column per year
ttc_wide = (
    ttc.pivot_table(index='poly_id', columns='pred_year', values='TTC', aggfunc='first')
)
ttc_wide.columns = [f'ttc_{int(yr)}' for yr in ttc_wide.columns]
ttc_wide = ttc_wide.reset_index()

# Drop the existing (empty) ttc columns from feats, then merge in the new ones
yr_cols = [c for c in feats.columns if c.startswith('ttc_')]
feats_base = feats.drop(columns=yr_cols)

result = feats_base.merge(ttc_wide, on='poly_id', how='left')

# Restore original feats column order
result = result[feats.columns]

#result.to_csv('tm_feats_c2_with_ttc.csv', index=False)

In [35]:
result.to_csv('../data/feats/tm_feats_c2_04-13-2026.csv', index=False)